# featurewiz, explained 1/4: Intuition
featurewiz picks features in two stages: **SULOV** throws out redundant features,
then **Recursive XGBoost** keeps the ones a model actually uses. Before opening
either stage, let's see the problem they solve, on data where we *know* the answer.

In [ ]:
import pathlib, sys

for module_file in pathlib.Path("/kaggle/input").glob("**/featurewiz_explained.py"):
    sys.path.insert(0, str(module_file.parent))
    break

import featurewiz_explained as fwx
import featurewiz_explained_plots as plots

> **featurewiz, explained**: a four-part series that rebuilds featurewiz 0.6.1's feature selection step by step.
> **1 Intuition** · 2 SULOV · 3 Recursive XGBoost · 4 Full pipeline.
> Everything here runs with Internet **off**. Results are checked against Reference runs of the real featurewiz 0.6.1.

## The Canonical dataset
We'll use this dataset in every Lesson. Its features were built on purpose, so each one has a
**planted role**:
- `x1`–`x4` are **informative**: the target depends on them.
- `x1_copy`–`x3_copy` are **redundant**: the original plus a little noise.
- `x4_squared` is **redundant** too, but a *nonlinear* copy. Remember it.
- `noise_1`–`noise_4` are **noise**: unrelated to the target.

In [ ]:
data = fwx.make_canonical_dataset()
plots.selection_table([], data).show()
data.frame.head()

## Redundant features show up as bright off-diagonal squares

In [ ]:
plots.correlation_heatmap(data.frame, data.features).show()

Each `x` and its `_copy` correlate above 0.9. `x4` and `x4_squared` barely correlate
linearly, even though one is computed from the other.

## Why bother? Redundancy splits importance
Train one XGBoost model on everything and look at how it spreads importance:

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import cross_val_score

model = xgb.XGBClassifier(n_estimators=200, max_depth=3, random_state=0, tree_method="exact")
model.fit(data.frame[data.features], data.frame[data.target])
gain = pd.Series(model.get_booster().get_score(importance_type="total_gain")).reindex(data.features, fill_value=0)
(gain / gain.sum()).round(3).sort_values(ascending=False).to_frame("share of gain")

Roughly **two fifths of the model's gain** lands on features that add nothing: the four copies take
about 22% between them and the four pure-noise columns about 21%. Each copy steals a little credit
from its original, and noise gets credit simply because trees can always find *some* split in it.
Feature selection should keep **one member of each Signal group** and drop the noise.

## Why bother? Noise and redundancy cost accuracy on small data

In [ ]:
small = data.frame.sample(300, random_state=0)
informative = [f for f in data.features if data.roles[f] == "informative"]
for name, cols in [("all 12 features", data.features), ("only the 4 informative", informative)]:
    scores = cross_val_score(
        xgb.XGBClassifier(n_estimators=200, max_depth=3, random_state=0, tree_method="exact"),
        small[cols], small[data.target], cv=5, scoring="roc_auc",
    )
    print(f"{name:>24}: AUC {scores.mean():.3f} ± {scores.std():.3f}")

On 300 rows the informative-only model scores a little higher (about 0.75 vs 0.73 AUC), but the
fold-to-fold spread (±0.04–0.06) is as big as the gap. Honest summary: dropping the 8 useless
features costs nothing and may help a bit; the clearer win is the **interpretability** above.

## What featurewiz does about it
The next three Lessons rebuild it piece by piece. Here's the whole thing in one call,
using our Re-implementation (checked against the real featurewiz 0.6.1):

In [ ]:
selection = fwx.select_features(data.frame, data.target)
print(f"{len(data.features)} features -> SULOV keeps {len(selection.sulov.kept)} -> Recursive XGBoost keeps {len(selection.selected)}")
plots.selection_table(selection.selected, data).show()

It kept `x1`, `x2` and `x3_copy` (a copy is fine: it carries the same signal), and **missed `x4`**.
Lesson 2 shows how SULOV chose between originals and copies. Lesson 3 shows why
Recursive XGBoost let `x4` slip away.